# Evaluating an agentic system

An agentic system is composed of several nodes, orchestrated in an intelligent manner. Nowadays, we implicitly design an agentic system as LLMs operating a logic through a graph of nodes. For this notebook, we use Langgraph as the orchestrator management system. Then, we can notably define nodes and links between them.

For this demonstration, we use small and local LLMs, served with ollama. The two models used to compare the performances regarding the evaluation of the agentic system are Qwen3.5 in its 2B version and Qwen3.5 in its 0.8B version.

## Import libraries

In [ ]:
import json
import pandas as pd
from jinja2 import Environment, FileSystemLoader
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.types import Command
from langchain_ollama import OllamaLLM, ChatOllama
from pydantic import BaseModel, Field

## Building the graph

The graph we build for this demonstration is applied to a sport event reservation chatbot system. From a request of the user (i.e. a question), a first node decides whether it is a `ticket_reservation` question or a `question_answering` related question. Then the concerned node produces an appropriate response.

We evaluate the system on three points:
- **final output accuracy** : is the final output appropriate ?
- **routing accuracy** : does the router redirect to the appropriate node ?
- **trajectory optimization** : is the trajectory taken by the system ideal ?

![Sports event reservation chatbot orchestration](architecture.svg)

In [ ]:
# first, we start to define a versatile llm agent

# MODEL_NAME = "qwen3.5:2b" # ! change model name here to use another one
MODEL_NAME = "qwen3.5:0.8b"

def ollama_llm(model_name=MODEL_NAME, temperature=0.6):
    return OllamaLLM(
        model=model_name,
        temperature=temperature, # temperature is not static since we need to be able to change it depending on the use-case (i.e. 0.0 for the router)
        reasoning=False # the model we chose is a reasoning model but we don't want the output to take time, so we disable it.
    ) 

In [ ]:
prompts = Environment(loader=FileSystemLoader("prompts"))
VALID_ROUTES = ("ticket_reservation", "question_answering")

def assistant_update(content: str) -> dict:
    return {"messages": [{"role": "assistant", "content": content}]}

# defining the nodes
def router(state: MessagesState):
    user_message = state['messages'][0].content

    llm = ollama_llm(temperature=0) # the router needs to be deterministic
    prompt = prompts.get_template("router.j2").render(user_message=user_message)
    llm_response = llm.invoke(prompt)

    route = json.loads(llm_response)["route"]
    if route not in VALID_ROUTES:
        raise ValueError(f"Unknown route: {route}")
    return Command(goto=route, update=assistant_update(llm_response))

def ticket_reservation(state: MessagesState):
    user_message = state['messages'][0].content
    llm = ollama_llm()
    prompt = prompts.get_template("ticket_reservation.j2").render(user_message=user_message)
    return assistant_update(llm.invoke(prompt))

def question_answering(state: MessagesState):
    user_message = state['messages'][0].content
    llm = ollama_llm()
    prompt = prompts.get_template("question_answering.j2").render(user_message=user_message)
    return assistant_update(llm.invoke(prompt))


# defining the whole graph
graph = StateGraph(MessagesState)
graph.add_node(router, destinations=VALID_ROUTES)
graph.add_node(ticket_reservation)
graph.add_node(question_answering)

graph.add_edge(START, "router")
graph.add_edge("ticket_reservation", END)
graph.add_edge("question_answering", END)

graph = graph.compile()

# testing the graph once
graph.invoke({
    "messages": [
        {
            'role': 'user',
            'content': 'What is the price for the game of the Eagles next Sunday ?'
        }
    ]
})


{'messages': [HumanMessage(content='What is the price for the game of the Eagles next Sunday ?', additional_kwargs={}, response_metadata={}, id='837553e4-2e15-4ce9-98f6-ebd8a9550e6e'),
  AIMessage(content='{"route": "ticket_reservation"}', additional_kwargs={}, response_metadata={}, id='1c2c7508-a04b-4c39-b343-694a93f588ac', tool_calls=[], invalid_tool_calls=[]),
  AIMessage(content='{"response": "The starting price for the Philadelphia Eagles vs Dallas Cowboys game on Sunday, August 23, 2026 at Lincoln Financial Field is $85."}', additional_kwargs={}, response_metadata={}, id='0c4fd77c-166b-4ed3-93cd-444d20385527', tool_calls=[], invalid_tool_calls=[])]}

## Evaluation

Evaluation lies in 3 steps:

1. Evaluating the routing accuracy

2. Evaluating the trajectory taken by the graph

3. Evaluating the accuracy of the model in answering the right question

For each step, we will be conducting an experiment using a small toy gold dataset.

> Note : we could use web-based tools like Langgraph Studio and Langsmith or even Langfuse (and more...) to submit evaluation jobs and follow the results on a nice UI but it is not the point of this notebook. I also think that the less the better and thus that a notebook is perfectly fine for this task.

### Question answering accuracy

Gold labels are grounded in a small event catalog. `answer` is the canonical reply; `key_points` are the facts a correct answer must include.

Accuracy is scored by an LLM-as-judge (`prompts/accuracy_judge.j2`), not exact string match, so paraphrases that preserve the facts still count as correct.

In [47]:
EVENTS = {
    "eagles_cowboys": {
        "home": "Philadelphia Eagles",
        "away": "Dallas Cowboys",
        "date": "Sunday, August 23, 2026",
        "time": "1:00 PM ET",
        "venue": "Lincoln Financial Field",
        "starting_price_usd": 85,
    },
    "yankees_red_sox": {
        "home": "New York Yankees",
        "away": "Boston Red Sox",
        "date": "Saturday, August 22, 2026",
        "time": "7:05 PM ET",
        "venue": "Yankee Stadium",
        "starting_price_usd": 45,
    },
    "lakers_warriors": {
        "home": "Los Angeles Lakers",
        "away": "Golden State Warriors",
        "date": "Friday, August 21, 2026",
        "time": "7:30 PM PT",
        "venue": "Crypto.com Arena",
        "starting_price_usd": 120,
    },
}

POLICIES = {
    "bag": "Clear bags only, maximum 12 x 6 x 12 inches.",
    "parking": "On-site parking is $40 per vehicle.",
    "children": "Children under 2 years old do not need a ticket.",
    "refunds": "Full refunds are available up to 24 hours before kickoff.",
}

qa_dataset = [
    {
        "id": "qa_001",
        "messages": [{"role": "user", "content": "What time does the Eagles game start next Sunday?"}],
        "answer": "The Philadelphia Eagles vs Dallas Cowboys game on Sunday, August 23, 2026 kicks off at 1:00 PM ET at Lincoln Financial Field.",
        "key_points": ["1:00 PM ET", "Dallas Cowboys", "Lincoln Financial Field"],
    },
    {
        "id": "qa_002",
        "messages": [{"role": "user", "content": "Where is the Yankees vs Red Sox game being played?"}],
        "answer": "The New York Yankees vs Boston Red Sox game on Saturday, August 22, 2026 is at Yankee Stadium.",
        "key_points": ["Yankee Stadium"],
    },
    {
        "id": "qa_003",
        "messages": [{"role": "user", "content": "How much do Lakers tickets start at for Friday's game?"}],
        "answer": "Tickets for the Los Angeles Lakers vs Golden State Warriors game on Friday, August 21, 2026 start at $120.",
        "key_points": ["$120", "Golden State Warriors"],
    },
    {
        "id": "qa_004",
        "messages": [{"role": "user", "content": "What is the price for the game of the Eagles next Sunday?"}],
        "answer": "Tickets for the Philadelphia Eagles vs Dallas Cowboys game on Sunday, August 23, 2026 start at $85.",
        "key_points": ["$85"],
    },
    {
        "id": "qa_005",
        "messages": [{"role": "user", "content": "What's the bag policy at Lincoln Financial Field?"}],
        "answer": "Clear bags only, maximum 12 x 6 x 12 inches.",
        "key_points": ["clear bags", "12 x 6 x 12"],
    },
    {
        "id": "qa_006",
        "messages": [{"role": "user", "content": "How much is parking at the venue?"}],
        "answer": "On-site parking is $40 per vehicle.",
        "key_points": ["$40"],
    },
    {
        "id": "qa_007",
        "messages": [{"role": "user", "content": "Do children under 2 need a ticket?"}],
        "answer": "Children under 2 years old do not need a ticket.",
        "key_points": ["under 2", "do not need a ticket"],
    },
    {
        "id": "qa_008",
        "messages": [{"role": "user", "content": "What's the refund policy if I can't attend?"}],
        "answer": "Full refunds are available up to 24 hours before kickoff.",
        "key_points": ["full refund", "24 hours"],
    },
]

In [ ]:
class JudgeOutput(BaseModel):
    score: int = Field(description="1 if the answer is accurate, 0 otherwise")
    reasoning: str = Field(description="Short explanation for the score")

def judge_answer_accuracy(
    user_message: str,
    gold_answer: str,
    key_points: list[str],
    predicted_answer: str,
) -> dict:
    llm = ChatOllama(model=MODEL_NAME, temperature=0, reasoning=False).with_structured_output(JudgeOutput)
    prompt = prompts.get_template("accuracy_judge.j2").render(
        user_message=user_message,
        gold_answer=gold_answer,
        key_points=key_points,
        predicted_answer=predicted_answer,
    )
    result = llm.invoke(prompt)
    return result.model_dump()


accuracy_results = []
for example in qa_dataset:
    prediction = graph.invoke({"messages": example["messages"]})
    predicted_answer = prediction['messages'][-1].content

    judge_answer = judge_answer_accuracy(
        user_message=example["messages"][0]["content"],
        gold_answer=example["answer"],
        key_points=example["key_points"],
        predicted_answer=predicted_answer,
    )

    accuracy_results.append({
        **example,
        "judge_label": judge_answer['score'],
        "judge_reasoning": judge_answer['reasoning']
    })

In [ ]:
df_accuracy = pd.DataFrame(accuracy_results)
df_accuracy.head()

,id,messages,answer,key_points,judge_label,judge_reasoning
0,qa_001,"[{'role': 'user', 'content': 'What time does t...",The Philadelphia Eagles vs Dallas Cowboys game...,"[1:00 PM ET, Dallas Cowboys, Lincoln Financial...",1,The predicted answer correctly identifies the ...
1,qa_002,"[{'role': 'user', 'content': 'Where is the Yan...",The New York Yankees vs Boston Red Sox game on...,[Yankee Stadium],1,The predicted answer correctly identifies the ...
2,qa_003,"[{'role': 'user', 'content': 'How much do Lake...",Tickets for the Los Angeles Lakers vs Golden S...,"[$120, Golden State Warriors]",1,The predicted answer correctly identifies the ...
3,qa_004,"[{'role': 'user', 'content': 'What is the pric...",Tickets for the Philadelphia Eagles vs Dallas ...,[$85],1,The predicted answer correctly identifies the ...
4,qa_005,"[{'role': 'user', 'content': 'What's the bag p...","Clear bags only, maximum 12 x 6 x 12 inches.","[clear bags, 12 x 6 x 12]",1,The predicted answer correctly identifies the ...


In [50]:
pd.DataFrame(df_accuracy['judge_label'].describe()).T[['mean', 'std']]

,mean,std
judge_label,1.0,0.0


### Trajectory optimization

Gold labels are the optimal node sequence through the graph.
In this orchestration that is always `router` followed by the correct specialist, then `END`.

In [31]:
trajectory_dataset = [
    {
        "id": "traj_001",
        "messages": [{"role": "user", "content": "I want to book 2 tickets for the Eagles game next Sunday."}],
        "trajectory": ["router", "ticket_reservation"],
    },
    {
        "id": "traj_002",
        "messages": [{"role": "user", "content": "Reserve 4 seats for Yankees vs Red Sox on Saturday."}],
        "trajectory": ["router", "ticket_reservation"],
    },
    {
        "id": "traj_003",
        "messages": [{"role": "user", "content": "Can you buy me a ticket for the Lakers game on Friday?"}],
        "trajectory": ["router", "ticket_reservation"],
    },
    {
        "id": "traj_004",
        "messages": [{"role": "user", "content": "Please cancel my reservation for the Warriors game."}],
        "trajectory": ["router", "ticket_reservation"],
    },
    {
        "id": "traj_005",
        "messages": [{"role": "user", "content": "Change my seats to section 112 for the Eagles game."}],
        "trajectory": ["router", "ticket_reservation"],
    },
    {
        "id": "traj_006",
        "messages": [{"role": "user", "content": "What time does the Eagles game start next Sunday?"}],
        "trajectory": ["router", "question_answering"],
    },
    {
        "id": "traj_007",
        "messages": [{"role": "user", "content": "Where is the Yankees vs Red Sox game being played?"}],
        "trajectory": ["router", "question_answering"],
    },
    {
        "id": "traj_008",
        "messages": [{"role": "user", "content": "What's the bag policy at Lincoln Financial Field?"}],
        "trajectory": ["router", "question_answering"],
    },
    {
        "id": "traj_009",
        "messages": [{"role": "user", "content": "What is the price for the game of the Eagles next Sunday?"}],
        "trajectory": ["router", "question_answering"],
    },
    {
        "id": "traj_010",
        "messages": [{"role": "user", "content": "What's the refund policy if I can't attend?"}],
        "trajectory": ["router", "question_answering"],
    },
]

In [32]:
# a function to retrieve the trajectory used by the graph
def get_trajectory(graph, messages):
    steps = []
    for event in graph.stream({"messages": messages}, stream_mode="updates"):
        steps.extend(event.keys())
    return steps

get_trajectory(graph, [
    {"role": "user", "content": "What is the price for the game of the Eagles next Sunday ?"}
])

['router', 'ticket_reservation']

In [33]:
# looping over the dataset
trajectories_results = []

for row in trajectory_dataset:
    messages = row['messages']
    trajectory = get_trajectory(graph, messages=messages)
    
    trajectories_results.append({
        **row,
        "predicted_trajectory": trajectory
    })

In [34]:
# calculating and displaying metrics
trajectories_results

[{'id': 'traj_001',
  'messages': [{'role': 'user',
    'content': 'I want to book 2 tickets for the Eagles game next Sunday.'}],
  'trajectory': ['router', 'ticket_reservation'],
  'predicted_trajectory': ['router', 'ticket_reservation']},
 {'id': 'traj_002',
  'messages': [{'role': 'user',
    'content': 'Reserve 4 seats for Yankees vs Red Sox on Saturday.'}],
  'trajectory': ['router', 'ticket_reservation'],
  'predicted_trajectory': ['router', 'ticket_reservation']},
 {'id': 'traj_003',
  'messages': [{'role': 'user',
    'content': 'Can you buy me a ticket for the Lakers game on Friday?'}],
  'trajectory': ['router', 'ticket_reservation'],
  'predicted_trajectory': ['router', 'ticket_reservation']},
 {'id': 'traj_004',
  'messages': [{'role': 'user',
    'content': 'Please cancel my reservation for the Warriors game.'}],
  'trajectory': ['router', 'ticket_reservation'],
  'predicted_trajectory': ['router', 'ticket_reservation']},
 {'id': 'traj_005',
  'messages': [{'role': 'user',

In [ ]:
df_trajectory = pd.DataFrame(trajectories_results)
df_trajectory

,id,messages,trajectory,predicted_trajectory
0,traj_001,"[{'role': 'user', 'content': 'I want to book 2...","[router, ticket_reservation]","[router, ticket_reservation]"
1,traj_002,"[{'role': 'user', 'content': 'Reserve 4 seats ...","[router, ticket_reservation]","[router, ticket_reservation]"
2,traj_003,"[{'role': 'user', 'content': 'Can you buy me a...","[router, ticket_reservation]","[router, ticket_reservation]"
3,traj_004,"[{'role': 'user', 'content': 'Please cancel my...","[router, ticket_reservation]","[router, ticket_reservation]"
4,traj_005,"[{'role': 'user', 'content': 'Change my seats ...","[router, ticket_reservation]","[router, ticket_reservation]"
5,traj_006,"[{'role': 'user', 'content': 'What time does t...","[router, question_answering]","[router, ticket_reservation]"
6,traj_007,"[{'role': 'user', 'content': 'Where is the Yan...","[router, question_answering]","[router, ticket_reservation]"
7,traj_008,"[{'role': 'user', 'content': 'What's the bag p...","[router, question_answering]","[router, ticket_reservation]"
8,traj_009,"[{'role': 'user', 'content': 'What is the pric...","[router, question_answering]","[router, ticket_reservation]"
9,traj_010,"[{'role': 'user', 'content': 'What's the refun...","[router, question_answering]","[router, ticket_reservation]"


In [36]:
# defining the scoring function
from difflib import SequenceMatcher

# we use the difflib sequence matcher because order and duplicates matter in agentic systems.
# in a bigger graph, the router could call a node more than once, going back and forth.
def trajectory_f(y_true, y_pred) -> float:
    """
    The function that scores the results for the trajectory task.

    y_true: the gold (targeted) trajectory
    y_pred: the predicted trajectory

    returns: a float, being the similarity between the gold and predicted trajectories.
    """
    return SequenceMatcher(None, y_true, y_pred).ratio()

# testing out the function
print(trajectory_f( # positive example
    y_true=["router", "ticket_reservation"],
    y_pred=["router", "ticket_reservation"],
))
print(trajectory_f( # negative example
    y_true=["router", "ticket_reservation"],
    y_pred=["router", "question_answering"],
)) 

1.0
0.5


In [37]:
# applying it to the dataframe
df_trajectory['score'] = df_trajectory.apply(lambda row: trajectory_f(row['trajectory'], row['predicted_trajectory']), axis=1)

In [38]:
df_trajectory['score'].describe()

count    10.000000
mean      0.750000
std       0.263523
min       0.500000
25%       0.500000
50%       0.750000
75%       1.000000
max       1.000000
Name: score, dtype: float64

In [39]:
pd.DataFrame(df_trajectory['score'].describe()).T[['mean', 'std']]

,mean,std
score,0.75,0.263523


So, over the trajectories, we obtain a mean score of 0.75 which means that the trajectories are 75% ideal.

### Routing accuracy

Gold labels are the subgraph the router should choose for each user request:

- `ticket_reservation`: the user wants to book, buy, change, or cancel tickets
- `question_answering`: the user wants information (schedule, price, venue, policy, etc.)

In [40]:
routing_dataset = [
    {
        "id": "route_001",
        "messages": [{"role": "user", "content": "I want to book 2 tickets for the Eagles game next Sunday."}],
        "route": "ticket_reservation",
    },
    {
        "id": "route_002",
        "messages": [{"role": "user", "content": "Reserve 4 seats for Yankees vs Red Sox on Saturday."}],
        "route": "ticket_reservation",
    },
    {
        "id": "route_003",
        "messages": [{"role": "user", "content": "Can you buy me a ticket for the Lakers game on Friday?"}],
        "route": "ticket_reservation",
    },
    {
        "id": "route_004",
        "messages": [{"role": "user", "content": "Please cancel my reservation for the Warriors game."}],
        "route": "ticket_reservation",
    },
    {
        "id": "route_005",
        "messages": [{"role": "user", "content": "Change my seats to section 112 for the Eagles game."}],
        "route": "ticket_reservation",
    },
    {
        "id": "route_006",
        "messages": [{"role": "user", "content": "What time does the Eagles game start next Sunday?"}],
        "route": "question_answering",
    },
    {
        "id": "route_007",
        "messages": [{"role": "user", "content": "Where is the Yankees vs Red Sox game being played?"}],
        "route": "question_answering",
    },
    {
        "id": "route_008",
        "messages": [{"role": "user", "content": "What's the bag policy at Lincoln Financial Field?"}],
        "route": "question_answering",
    },
    {
        "id": "route_009",
        "messages": [{"role": "user", "content": "What is the price for the game of the Eagles next Sunday?"}],
        "route": "question_answering",
    },
    {
        "id": "route_010",
        "messages": [{"role": "user", "content": "What's the refund policy if I can't attend?"}],
        "route": "question_answering",
    },
]

In [41]:
# scoring function for this routing evaluation
def routing_f(y_true, y_pred) -> float:
    """
    The function that scores the results for the routing task.

    y_true: the gold route
    y_pred: the predicated route

    returns: a float, being 1 if the route is accurate, 0 else.
    """
    return float(y_true == y_pred)

# testing the function
print(routing_f(y_true="question_answering", y_pred="ticket_reservation")) # 0.0
print(routing_f(y_true="question_answering", y_pred="question_answering")) # 1.0

0.0
1.0


In [42]:
# looping over the dataset
routing_results = []

for row in routing_dataset:
    messages = row['messages']
    trajectory = get_trajectory(graph, messages=messages) # re-using the trajectory function to get the next node after routing
    
    routing_results.append({
        **row,
        "trajectory": trajectory
    })

In [43]:
df_routing = pd.DataFrame(routing_results)
df_routing.head()

,id,messages,route,trajectory
0,route_001,"[{'role': 'user', 'content': 'I want to book 2...",ticket_reservation,"[router, ticket_reservation]"
1,route_002,"[{'role': 'user', 'content': 'Reserve 4 seats ...",ticket_reservation,"[router, ticket_reservation]"
2,route_003,"[{'role': 'user', 'content': 'Can you buy me a...",ticket_reservation,"[router, ticket_reservation]"
3,route_004,"[{'role': 'user', 'content': 'Please cancel my...",ticket_reservation,"[router, ticket_reservation]"
4,route_005,"[{'role': 'user', 'content': 'Change my seats ...",ticket_reservation,"[router, ticket_reservation]"


In [44]:
def retrieve_node_after_routing(trajectory):
    for i, node in enumerate(trajectory):
        if i != 0:
            if trajectory[i-1] == "router":
                return node
    return None

df_routing['predicted_route'] = df_routing['trajectory'].apply(retrieve_node_after_routing)
df_routing.head()

,id,messages,route,trajectory,predicted_route
0,route_001,"[{'role': 'user', 'content': 'I want to book 2...",ticket_reservation,"[router, ticket_reservation]",ticket_reservation
1,route_002,"[{'role': 'user', 'content': 'Reserve 4 seats ...",ticket_reservation,"[router, ticket_reservation]",ticket_reservation
2,route_003,"[{'role': 'user', 'content': 'Can you buy me a...",ticket_reservation,"[router, ticket_reservation]",ticket_reservation
3,route_004,"[{'role': 'user', 'content': 'Please cancel my...",ticket_reservation,"[router, ticket_reservation]",ticket_reservation
4,route_005,"[{'role': 'user', 'content': 'Change my seats ...",ticket_reservation,"[router, ticket_reservation]",ticket_reservation


In [45]:
df_routing['routing_score'] = df_routing.apply(lambda row: routing_f(y_true=row['route'], y_pred=row['predicted_route']), axis=1)
df_routing['routing_score'].describe()

count    10.000000
mean      0.500000
std       0.527046
min       0.000000
25%       0.000000
50%       0.500000
75%       1.000000
max       1.000000
Name: routing_score, dtype: float64

In [46]:
pd.DataFrame(df_routing['routing_score'].describe()).T[['mean', 'std']]

,mean,std
routing_score,0.5,0.527046


In this case the routing accuracy is the same as the trajectory accuracy because we have only one step after the router. However, in a bigger system, these two evaluation steps would mean much more.

# Conclusion

## Trajectory

The trajectory predicted is not always the ideal even though the graph is really simple (2 nodes depth at the most). We tested with the qwen3.5 model in its 0.8B version and we detail more in the next section. The **final mean score of the trajectory is 0.9 on average**.

## Routing

With the 0.8B version of the qwen3.5 model, the results were poor, the model always routing towards the `question_anwering` node. Switching to the 2B version of the same model has improved a lot the routing, **climbing up from a mean of 0.5 to a mean of 0.8**.

## Accuracy of the answer

The accuracy of the answer is simply answering the question : does the graph answers the correct information to the user ?

A barrier was that typical user questions are asking for detailed information that the model doesn't have. In a real-world application, we would setup architectures like RAG (Retrieval-Augmented Generation) to provide the model with accurate and relevant answers but this is a tutorial in which we can't afford to complicate the setup. We just embedded this information in the prompt.

The graph is thus generating the final answer, and a judge LLM compares the generated answer with the gold answer. We end up scoring a **mean score of 1**, meaning that the graph always answers something relevant to the user, even with a 2B qwen3.5 model.

# References

Here are the references I used to build this ressource.

- [Beginner's Guide to Agent Evaluations - Langchain on YouTube](https://www.youtube.com/watch?v=_QozKR9eQE8)

- [Agents Course - HuggingFace](https://huggingface.co/learn/agents-course/unit0/introduction)

- [The Langgraph documentation](https://docs.langchain.com/oss/python/langgraph/overview)